# 06 - Direct calls across the entire available period
Uses every supplied date from January 1, 2024 through October 16, 2026: the early raw CAD log and the subsequent published retained-call extract. Matching uses the fixed 4,806-parcel residential cohort. Call numbers are deduplicated, and conflicting repeated identifiers are audited. The later extract remains filtered/retained data; this combination does not establish complete category coverage throughout the span.

Parking is an explicit starred exception: its rates use the entire supported January 1-September 20, 2024 window, because later parking coverage is absent. Never divide early-only parking counts by the longer main denominator or interpret missing later parking as zero. Both rental and residential parking rates use the same early-period exposure. The report's default parking setting always plots all matched parcels, including zeros, regardless of an above-average main cohort. A second option follows that main cohort. Selected initiation is never silently changed. Every parking graph has * and a settings/date footnote.

The source share has 4,561 listing folders, with 4,372 readable detail files. Known eligible permit/parcel matches remain required. No property is inferred from approximate Airbnb pins. Current header sizes have the old audited values as fallback. Amenities retain context labels and source hashes; restrictions are separate from on-site candidates.

Adjusted annual rates are direct calls / category exposure years / 0.59. The residential baseline is calendar-year calls over the same category exposure. 59% remains a scenario, not property-specific observed occupancy. Above-average selection is recomputed over the entire main span. Group means count each parcel once, including zeros, using maximum known listing size per parcel. Minimum-size curves reproduce the original map slider question, alongside exact-size groups. Disturbance / nuisance retains the published taxonomy.

Raw-source normalization and the full CAD coordinate match audit must be refreshed first if source workbooks or the digest change. The combined source hashes and date coverage are saved in processed_data/full_period_calls/coverage.json. Source details can be refreshed using the optional scan below; subsequent runs work offline.

## Live license and listing controls
The report includes the complete current workbook in the address audit, with an all-address or matched-Airbnb cohort selector. Calls remain unavailable for addresses outside the established residential study coverage, rather than becoming invented zeros. Graphs deduplicate parcels; approved guest capacity and licensed bedrooms sum unique license records on that parcel. Address rows preserve each certificate's own approved figures. Missing advertised data are omitted from advertised-size graphs only.

Over-limit screening compares current explicit advertised guests with the recorded approved capacity of the certificates actually matched to that listing. No universal bedroom formula is applied. The amenity checkboxes support OR/AND in the same listing and retain on-site, mention and restriction modes. All live controls change graphs and address rows together. Dynamic SVG and CSV exports preserve current settings. The starred parking override remains explicitly disclosed.


Two generated pages keep explicit capacity rules active: at_or_above_approved.html selects advertised >= approved, while above_approved.html selects advertised > approved, including +1. Both default to a new advertised-minus-approved size axis. Odd advertised values stay unrounded and flagged; oddness alone is not a compliance finding. Parking fallback and Clear filters retain the page rule. The generator saves both full qualifying-listing CSVs and capacity_page_summary.json.


Permit categories are read explicitly from Owner Occupied vs Non-Owner Occupied: Type 1 = owner, Type 2 = non_owner. Charts use separate mutually exclusive parcel groups; mixed-type parcels do not enter either pure category. The permit filter remains active for parking fallback, page capacity rules and exports. This category is distinct from the booking-occupancy scenario.


Source-match correction (October 28, 2026): historical filename-only joins were withdrawn. The active cohort requires an SVR code in source text, an uncontradicted displayed location and an unchanged audited source hash. The corrected snapshot has 758 listings on 484 parcels. Old executed outputs were cleared because they described superseded joins. See processed_data/listing_matches/ for the original join snapshot, full source audit and reversible folder rename log.


In [ ]:
from pathlib import Path
import sys, runpy
import pandas as pd
from IPython.display import display, Image
ROOT=Path.cwd().resolve()
if not (ROOT/'raw_data').is_dir(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
REFRESH_NETWORK_DETAILS=False
if REFRESH_NETWORK_DETAILS:
    runpy.run_module('processing.scan_current_listing_details',run_name='__main__')
from processing.direct_listing_analysis import run
summary=run(ROOT)
display(summary)

In [ ]:
rates=pd.read_csv(ROOT/'direct_listing_calls/citizen_parcel_direct_rates.csv')
above=pd.read_csv(ROOT/'direct_listing_calls/citizen_above_average_listings.csv',dtype={'listing_id':str,'parcel_PIN':str})
assert rates.pin.is_unique
assert len(rates)==summary['matched_parcels']
assert rates.above_average.sum()==summary['measures']['citizen']['above_average_parcels']
assert above.parcel_PIN.nunique()==rates.above_average.sum()
assert (rates['All calls_adjusted_rate']*0.59-rates['All calls_calendar_rate']).abs().max()<1e-10
assert rates['Parking complaints_count'].notna().all()
assert rates['Parking complaints_count'].sum()>0
assert summary['date_start']=='2024-01-01'
assert summary['date_end']=='2026-10-16'
parking_years=summary['call_coverage']['parking_exposure_years']
assert (rates['Parking complaints_calendar_rate']*parking_years-rates['Parking complaints_count']).abs().max()<1e-10
from pathlib import Path
for measure in ['citizen']:
    for size in ['bedrooms','guests']:
        for cohort in ['all_matched','above_average']:
            assert (ROOT/'plots'/'README.md').exists()
print('Unique-parcel, source call-count, threshold and occupancy-scaling checks passed.')
display(above[['title','parcel_PIN','bedrooms','guests','All calls_count','All calls_adjusted_rate','on_site_feature_candidates','active_mention_features','restriction_mentions']])

In [ ]:
features=pd.read_csv(ROOT/'direct_listing_calls/amenity_comparisons.csv')
display(features[features.measure.eq('citizen') & features.evidence_mode.eq('onsite')])
print('Individual citizen-only plots: plots/README.md')

In [ ]:
import json
payload=json.loads((ROOT/'direct_listing_calls/report_data.js').read_text(encoding='utf-8').removeprefix('window.DIRECT_REPORT=').removesuffix(';'))
controls=payload['controls']
properties=controls['properties']
assert len({p['pin'] for p in properties})==len(properties)
assert len(controls['license_addresses'])==controls['counts']['complete_license_rows']
license_audit=pd.read_csv(ROOT/'direct_listing_calls/complete_license_address_audit.csv')
assert abs(sum(p['approved_guests'] for p in properties)-license_audit.loc[license_audit.call_coverage_available,'approved_guests'].sum())<1e-10
assert abs(sum(p['licensed_bedrooms'] for p in properties)-license_audit.loc[license_audit.call_coverage_available,'licensed_bedrooms'].sum())<1e-10
for property in properties:
    for listing in property['listings']:
        if listing['over_limit']:
            assert listing['guests']>listing['approved_guests']
            assert listing['overage']==listing['guests']-listing['approved_guests']
print('Complete-address, deduplicated license totals and over-limit checks passed.')
display(controls['counts'])